In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Đọc và gộp dữ liệu Train (2022-2024)
files = ['../data_raw/2022.csv', '../data_raw/2023.csv', '../data_raw/2024.csv']
df_list = [pd.read_csv(f) for f in files]
df_train = pd.concat(df_list, axis=0, ignore_index=True)

# 2. Lấy cột số và xóa dòng trống (NaN)
X = df_train.select_dtypes(include=[np.number]).dropna()
X_values = X.values 

# --- BẮT ĐẦU DÁN TỪ ĐÂY ---

# 3. Chuẩn hóa thủ công (Standardization)
X_mean = np.mean(X_values, axis=0)
X_std = np.std(X_values, axis=0)
# Thêm 1 chút epsilon 1e-8 để tránh lỗi chia cho 0 nếu có cột nào đó toàn giá trị giống nhau
X_scaled = (X_values - X_mean) / (X_std + 1e-8)

# 4. Tính ma trận Hiệp phương sai
cov_matrix = np.cov(X_scaled.T)

# 5. Tính Trị riêng & Vectơ riêng
eigen_values, eigen_vectors = np.linalg.eig(cov_matrix)

# 6. Sắp xếp và lấy PHẦN THỰC (np.real) để tránh lỗi số phức khi vẽ hình
sorted_index = np.argsort(eigen_values)[::-1]
# Lấy phần thực để biểu đồ không bị lỗi
sorted_eigenvalues = np.real(eigen_values[sorted_index])

exp_var = sorted_eigenvalues / np.sum(sorted_eigenvalues)
cum_var = np.cumsum(exp_var)

# 7. Vẽ biểu đồ Scree Plot
plt.figure(figsize=(8, 5))
plt.bar(range(1, len(exp_var) + 1), exp_var, alpha=0.5, label='Thông tin riêng lẻ')
plt.step(range(1, len(cum_var) + 1), cum_var, where='mid', label='Thông tin tích lũy', color='red')
plt.axhline(y=0.88, color='g', linestyle='--', label='Mục tiêu 88%')
plt.ylabel('Tỷ lệ phương sai')
plt.xlabel('Thành phần chính')
plt.title('Scree Plot - PCA Thủ công')
plt.legend()
plt.show()

# 8. Nhận xét
info_2 = cum_var[1] * 100
print(f"Sử dụng PCA giúp giảm số biến từ {X.shape[1]} xuống 2 nhưng vẫn giữ được {info_2:.2f}% lượng thông tin, giúp mô hình của Phương Anh chạy nhanh và mượt hơn.")